# Кандидатогенерация для поиска услуг Avito

Задача. Для каждого поискового запроса отобрать до 50 объявлений-кандидатов из корпуса в
189 212 объявлений. Метрика — Recall@50, усреднённая по запросам доля найденных релевантных
объявлений.

Что получилось в итоге: Recall@50 = 0.8194 

1. BM25F по трём текстовым полям объявления с лемматизацией;
2. мягкий гео-приор по расстоянию;
3. двухканальный отбор кандидатов — глобальный поиск плюс поиск, ограниченный окрестностью
   запроса.

In [ ]:
import re
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import CountVectorizer

import pymorphy3

np.random.seed(42)
DATA_DIR = Path("data")
TOKEN_RE = re.compile(r"[0-9a-zа-яё]+")

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

## 1. Знакомство с данными

Даны три таблицы: обучающая выборка кликов, целевые запросы бенчмарка и целевой корпус
объявлений. Посмотрим, что в них лежит.

In [2]:
train = pq.read_table(DATA_DIR / "train.parquet").to_pandas()
bq = pq.read_table(DATA_DIR / "benchmark_queries.parquet").to_pandas()
bi = pq.read_table(DATA_DIR / "benchmark_items.parquet").to_pandas()

print(f"train            : {train.shape[0]:,} строк, {train.shape[1]} колонок")
print(f"benchmark_queries: {bq.shape[0]:,} строк, {bq.shape[1]} колонок")
print(f"benchmark_items  : {bi.shape[0]:,} строк, {bi.shape[1]} колонок")
print()
print("колонки train:", list(train.columns))
bq.head()

train            : 497,673 строк, 19 колонок
benchmark_queries: 2,452 строк, 6 колонок
benchmark_items  : 189,212 строк, 14 колонок

колонки train: ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category', 'item_title_raw', 'item_rating_reviews_count', 'item_rating', 'item_price', 'item_microcat_id', 'item_longitude', 'item_location_id', 'item_latitude', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_infm_params_text', 'item_id', 'item_description_raw', 'item_category_id']


,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114
3,660ac9QVtXkRxZC3,подъемник 4 х стоечный,662810,0,,114
4,YgHcM9MVbxKnxD1e,монтаж видеодомофонов,642790,0,,114


В train лежат пары «запрос — кликнутое объявление»: колонки с префиксом search_ описывают
запрос, с префиксом item_ — объявление. Отдельного query_id в train нет, запрос задаётся
тройкой (search_query, search_location_id, search_category).

Посмотрим на объявление.

In [3]:
with pd.option_context("display.max_colwidth", 90):
    display(bi[["item_id", "item_title_raw", "item_location_id",
                "item_latitude", "item_longitude", "item_price", "item_rating"]].head())

print("пример полного объявления")
row = bi.iloc[0]
print("заголовок   :", row.item_title_raw)
print("параметры   :", str(row.item_infm_params_text)[:300], "...")
print("описание    :", str(row.item_description_raw)[:300], "...")

,item_id,item_title_raw,item_location_id,item_latitude,item_longitude,item_price,item_rating
0,111eb8b979577d79,Ремонт/выкуп компьют. и ноутбуков с выездом на дом,631060,44.228180450924000,42.047193961004901,500.000000000000000,5.0
1,75fc8e10f5a66fc4,Обучение ребенка чтению,631870,58.627609249999999,49.627391820000000,900.000000000000000,5.0
2,3f6ae81704565b9c,Афрокудри 5+,628500,56.990307000000001,40.936892000000000,1000.000000000000000,5.0
3,0618dec37a59a21a,Монтаж малых архитектурных форм,637640,55.662734999999998,37.576183000000000,5000.000000000000000,5.0
4,13da2c81574677ed,Репетитор по истории 10 класс ЕГЭ,624850,48.786008000000002,44.750473999999997,1500.000000000000000,5.0


пример полного объявления
заголовок   : Ремонт/выкуп компьют. и ноутбуков с выездом на дом
параметры   : Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от 34200 График работы до 82800 Время работы, с 09:30 Время работы, до 23:00 Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Диагностика компьютера Начальная цена Тип стои ...
описание    : Выезд на дом в любое время, вплоть до 23:00

● Диагностика устройств;

● Чистка от пыли и замена термопасты;

● Проверка работоспособности всех компонентов.

Ремонт аппаратной части:

● Замена и ремонт материнской платы;

● Ремонт и замена жесткого диска (HDD/SSD);

● Установка и замена оперативной  ...


Текст объявления разложен на три поля: короткий заголовок, структурированные параметры
(«Вид услуги», «Стоимость» и подобные) и длинное свободное описание. Параметры оказались не
служебными метаданными, а полноценным текстом — по словарю они пересекаются с фильтрами запроса
search_infm_params_text.

Проверим, какие поля вообще несут информацию.

In [4]:
for col in ["search_category", "item_category_id"]:
    top = train[col].value_counts(normalize=True).head(2)
    print(f"{col}: доминирующее значение {top.index[0]} занимает {top.iloc[0]:.1%} строк")

print()
print("доля пустых фильтров запроса:")
print(f"  train    : {train.search_infm_params_text.fillna('').eq('').mean():.1%}")
print(f"  benchmark: {bq.search_infm_params_text.fillna('').eq('').mean():.1%}")

overlap = len(set(bi.item_id) & set(train.item_id)) / len(bi)
print()
print(f"доля benchmark_items, встречающихся в train: {overlap:.1%}")

search_category: доминирующее значение 114 занимает 100.0% строк
item_category_id: доминирующее значение 114 занимает 100.0% строк

доля пустых фильтров запроса:
  train    : 33.0%
  benchmark: 63.1%



доля benchmark_items, встречающихся в train: 9.6%


Три вывода, которые сразу отсекают часть подходов.

search_category и item_category_id заняты одним значением почти на всех строках — это
константа. Дальше их не используем.

Фильтры запроса заполнены лишь у трети запросов бенчмарка, так что опираться только на них
нельзя.

Лишь около 10% объявлений целевого корпуса встречаются в train. Значит подходы,
основанные на запоминании (популярность объявления, совстречаемость «запрос — объявление»),
имеют низкий потолок по построению — для 90% корпуса истории кликов просто нет. 

## 2. Что связывает запрос и объявление

Прежде чем строить поиск, посмотрим на асимметрию длин и на то, какие поля объявления реально
покрывают слова запроса.

In [5]:
q_words = train.search_query.fillna("").str.split().str.len()
print("длина запроса в словах:")
print(f"  медиана {q_words.median():.0f}, среднее {q_words.mean():.2f}, "
      f"доля однословных {(q_words == 1).mean():.1%}")

for col, name in [("item_title_raw", "заголовок"),
                  ("item_infm_params_text", "параметры"),
                  ("item_description_raw", "описание")]:
    lens = train[col].fillna("").str.len()
    print(f"длина поля '{name}' в символах: медиана {lens.median():.0f}, "
          f"95-й перцентиль {lens.quantile(0.95):.0f}")

длина запроса в словах:
  медиана 2, среднее 2.34, доля однословных 19.8%
длина поля 'заголовок' в символах: медиана 34, 95-й перцентиль 50


длина поля 'параметры' в символах: медиана 871, 95-й перцентиль 2965


длина поля 'описание' в символах: медиана 920, 95-й перцентиль 3876


Запрос — это два-три слова, описание — около тысячи символов. Разрыв примерно в пятьдесят раз.
Из этого следует, что нескольких терминов запроса не хватит, чтобы развести сотни похожих
объявлений услуг между собой.

Теперь проверим, какое поле вообще содержит слова запроса. Считаем на выборке пар из train
долю слов запроса, встретившихся в полях кликнутого объявления.

In [6]:
morph = pymorphy3.MorphAnalyzer()
_lemma_cache = {}


def lemmas(text):
    out = []
    for token in TOKEN_RE.findall(text):
        value = _lemma_cache.get(token)
        if value is None:
            value = morph.parse(token)[0].normal_form
            _lemma_cache[token] = value
        out.append(value)
    return out


sample = train.sample(30_000, random_state=SEED)
fields = {
    "только заголовок": ["item_title_raw"],
    "только описание": ["item_description_raw"],
    "заголовок+параметры": ["item_title_raw", "item_infm_params_text"],
    "все три поля": ["item_title_raw", "item_infm_params_text", "item_description_raw"],
}

rows = []
for name, cols in fields.items():
    cover, empty = [], 0
    for r in sample.itertuples():
        q = set(TOKEN_RE.findall(str(r.search_query).lower()))
        if not q:
            continue
        doc = " ".join(str(getattr(r, c) or "") for c in cols).lower()
        d = set(TOKEN_RE.findall(doc))
        share = len(q & d) / len(q)
        cover.append(share)
        empty += share == 0
    rows.append({"поля": name, "среднее покрытие": round(float(np.mean(cover)), 3),
                 "нет общих слов": f"{empty / len(cover):.1%}"})

print(pd.DataFrame(rows).to_string(index=False))

               поля  среднее покрытие нет общих слов
   только заголовок             0.552          27.7%
    только описание             0.638          20.6%
заголовок+параметры             0.676          15.7%
       все три поля             0.825           6.9%


In [7]:
cover_raw, cover_lem, empty_raw, empty_lem = [], [], 0, 0
for r in sample.head(8000).itertuples():
    q_raw = set(TOKEN_RE.findall(str(r.search_query).lower()))
    if not q_raw:
        continue
    title = str(r.item_title_raw or "").lower()
    d_raw = set(TOKEN_RE.findall(title))
    q_lem, d_lem = set(lemmas(str(r.search_query).lower())), set(lemmas(title))
    s_raw, s_lem = len(q_raw & d_raw) / len(q_raw), len(q_lem & d_lem) / len(q_lem)
    cover_raw.append(s_raw)
    cover_lem.append(s_lem)
    empty_raw += s_raw == 0
    empty_lem += s_lem == 0

n = len(cover_raw)
print("покрытие слов запроса заголовком")
print(f"  точное совпадение   : {np.mean(cover_raw):.3f}, без общих слов {empty_raw / n:.1%}")
print(f"  после лемматизации  : {np.mean(cover_lem):.3f}, без общих слов {empty_lem / n:.1%}")

покрытие слов запроса заголовком
  точное совпадение   : 0.554, без общих слов 27.5%
  после лемматизации  : 0.651, без общих слов 16.7%


Заголовок покрывает лишь чуть больше половины слов запроса, и у заметной доли пар с заголовком
вообще нет общих слов. Значит индексировать нужно все три поля, а не только заголовок — иначе часть релевантных
объявлений недостижима.

Лемматизация заметно сокращает долю пар без
пересечения. Для русского языка с его словоизменением это ожидаемо, и стоит это дёшево, если
кэшировать разбор по уникальным токенам.

Даже по всем трём полям остаётся около 17% слов запроса, которым в объявлении
нет соответствия. 

## 3. География

У запроса и у объявления есть локация и координаты. Услуги оказываются на месте, поэтому логично
ожидать, что пользователь кликает на что-то рядом. Проверим, насколько жёстко это выполняется.

In [8]:
same_loc = (train.search_location_id == train.item_location_id).mean()
print(f"доля кликов, где локация запроса совпала с локацией объявления: {same_loc:.1%}")

share_in_loc = bq.search_location_id.map(
    bi.item_location_id.value_counts(normalize=True)).fillna(0)
print(f"медианная доля корпуса в локации запроса: {share_in_loc.median():.2%} "
      f"(~{int(share_in_loc.median() * len(bi)):,} объявлений)")
print(f"запросов, чьей локации нет ни у одного объявления корпуса: "
      f"{(share_in_loc == 0).mean():.1%}")

доля кликов, где локация запроса совпала с локацией объявления: 83.1%
медианная доля корпуса в локации запроса: 0.87% (~1,638 объявлений)
запросов, чьей локации нет ни у одного объявления корпуса: 17.4%


Совпадение локаций на уровне 83% — сигнал очень сильный, но не абсолютный.
Будем использовать мягкий приор — надбавку к скору, убывающую с расстоянием.

Второе наблюдение: у 17% запросов бенчмарка локация вообще отсутствует в корпусе объявлений.
Такие запросы не получат гео-сигнала, и с ними придётся разбираться отдельно.

## 4. Валидационный стенд

Чтобы проверять гипотезы, не расходуя попытки отправки, нужен офлайн-стенд. Сделаем его так:
отложим часть запросов train (их клики станут истиной) и будем искать по настоящему корпусу
benchmark_items, дополненному отложенными позитивами — иначе метрику нечем было бы измерить.

Кроме состава корпуса выравниваем ещё две вещи — распределение локаций запросов и распределение
частот запросов, поскольку бенчмарк заметно более «хвостовой», чем train.

In [9]:
KEY = ["search_query", "search_location_id", "search_category"]
qid_codes, qid_uniques = pd.factorize(pd.MultiIndex.from_frame(train[KEY]))
train["qid"] = qid_codes
N_VAL = 2452

bench_loc_freq = bq.search_location_id.value_counts(normalize=True)
grp_loc = train.drop_duplicates("qid").set_index("qid").search_location_id
groups_by_loc = grp_loc.groupby(grp_loc).groups

qfreq = train.search_query.value_counts()
seen_freqs = bq.search_query.map(qfreq).dropna()


def bucket(f):
    if f <= 1:
        return 0
    if f <= 3:
        return 1
    if f <= 8:
        return 2
    if f <= 22:
        return 3
    return 4


seen_buckets = pd.Series([bucket(f) for f in seen_freqs]).value_counts(normalize=True)
target = {b: 0.37 * seen_buckets.get(b, 0.0) for b in range(5)}
target[0] += 0.63
total = sum(target.values())
target = {b: v / total for b, v in target.items()}

grp_qtext = train.drop_duplicates("qid").set_index("qid").search_query
grp_bucket = grp_qtext.map(qfreq).fillna(1).map(bucket)

rng = np.random.RandomState(SEED)
picked = []
for loc, freq in bench_loc_freq.items():
    need = int(round(freq * N_VAL))
    if need == 0:
        continue
    avail = np.array(groups_by_loc.get(loc, []))
    if len(avail) == 0:
        continue
    av_b = grp_bucket.reindex(avail).values
    chosen = []
    for b in range(5):
        want = int(round(target[b] * need))
        pool_b = avail[av_b == b]
        if len(pool_b) and want:
            chosen.extend(rng.choice(pool_b, size=min(want, len(pool_b)), replace=False).tolist())
    if len(chosen) < need:
        rest = np.array([q for q in avail if q not in set(chosen)])
        if len(rest):
            chosen.extend(rng.choice(rest, size=min(need - len(chosen), len(rest)),
                                     replace=False).tolist())
    picked.extend(chosen[:need])

if len(picked) < N_VAL:
    pool = [q for loc in bench_loc_freq.index
            for q in groups_by_loc.get(loc, []) if q not in set(picked)]
    if pool:
        picked.extend(rng.choice(np.array(pool), size=min(N_VAL - len(picked), len(pool)),
                                 replace=False).tolist())

val_set = set(int(q) for q in picked)
val_rows = train[train.qid.isin(val_set)]
positives = val_rows.groupby("qid").item_id.apply(set)
q_attrs = val_rows.drop_duplicates("qid").set_index("qid")
val_qids = np.array(sorted(val_set))

VAL_TRUTH = [positives.loc[q] for q in val_qids]
VAL_QTEXT = q_attrs.loc[val_qids, "search_query"].values
VAL_QLOC = q_attrs.loc[val_qids, "search_location_id"].values
VAL_QPARAMS = q_attrs.loc[val_qids, "search_infm_params_text"].fillna("").values
must_have = set().union(*VAL_TRUTH)
print(f"отложено запросов: {len(val_qids):,} | уникальных позитивов: {len(must_have):,}")

отложено запросов: 2,452 | уникальных позитивов: 2,707


In [10]:
KEEP = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
        "item_microcat_id", "item_location_id", "item_latitude", "item_longitude"]
inject = (train[train.item_id.isin(must_have - set(bi.item_id))]
          .drop_duplicates("item_id")[KEEP])

corpus = pd.concat([bi[KEEP], inject], ignore_index=True).drop_duplicates("item_id")
corpus = corpus.reset_index(drop=True)
for c in ["item_latitude", "item_longitude"]:
    corpus[c] = pd.to_numeric(corpus[c], errors="coerce")
ROW2ITEM = corpus.item_id.values
assert must_have <= set(corpus.item_id)

share_stand = pd.Series(VAL_QLOC).map(
    corpus.item_location_id.value_counts(normalize=True)).fillna(0)
share_real = bq.search_location_id.map(bi.item_location_id.value_counts(normalize=True)).fillna(0)
print(f"корпус стенда: {len(corpus):,} объявлений "
      f"(реальных {len(bi):,} + инъекция {len(corpus) - len(bi):,})")
print()
print("медианная доля корпуса в локации запроса")
print(f"  стенд            : {share_stand.median():.2%}")
print(f"  реальный бенчмарк: {share_real.median():.2%}")
print(f"  запросов без локации в корпусе: стенд {(share_stand == 0).mean():.1%}, "
      f"бенчмарк {(share_real == 0).mean():.1%}")


корпус стенда: 191,713 объявлений (реальных 189,212 + инъекция 2,501)

медианная доля корпуса в локации запроса
  стенд            : 0.87%
  реальный бенчмарк: 0.87%
  запросов без локации в корпусе: стенд 17.4%, бенчмарк 17.4%


Гео-окружение стенда совпало с реальным: и медианная доля корпуса в локации запроса, и доля
запросов без локации. 

## 5. Лексический поиск: BM25F

Строим индекс по трём полям с раздельными весами. BM25F отличается от простого BM25 тем, что
нормировка на длину выполняется внутри каждого поля отдельно, а насыщение применяется к сумме
взвешенных частот — иначе длинное описание забивало бы короткий заголовок.

Веса полей и параметры k1, b подобраны развёрткой (заголовок важнее всего, описание —
вспомогательное поле). Развёртка b вниз, вплоть до 0.3 по описанию, проверялась отдельно и
результат только ухудшала, поэтому здесь оставлены найденные значения.

In [11]:
def lemma_analyzer(doc):
    return lemmas(doc)


def build_index(frame):
    title = frame.item_title_raw.fillna("").str.lower()
    params = frame.item_infm_params_text.fillna("").str.lower()
    desc = frame.item_description_raw.fillna("").str.lower()
    vec = CountVectorizer(analyzer=lemma_analyzer, token_pattern=None, min_df=2)
    vec.fit(title + " " + title + " " + title + " " + params + " " + desc)
    counts = [vec.transform(title), vec.transform(params), vec.transform(desc)]
    n_doc = counts[0].shape[0]
    df_any = np.asarray(((counts[0] > 0) + (counts[1] > 0) + (counts[2] > 0)).sum(axis=0)).ravel()
    idf = np.log(1.0 + (n_doc - df_any + 0.5) / (df_any + 0.5)).astype(np.float32)
    return vec, counts, idf


def bm25f_matrix(counts, weights, idf, k1=0.8, b_per_field=(0.8, 0.8, 0.8)):
    combined = None
    for cf, wf, bf in zip(counts, weights, b_per_field):
        cf = cf.tocsr().astype(np.float32)
        dl = np.asarray(cf.sum(axis=1)).ravel()
        avgdl = max(dl[dl > 0].mean() if (dl > 0).any() else 1.0, 1.0)
        rows_ = np.repeat(np.arange(cf.shape[0], dtype=np.int32), np.diff(cf.indptr))
        norm = 1.0 - bf + bf * dl[rows_] / avgdl
        cf = cf.copy()
        cf.data = wf * cf.data / norm
        combined = cf if combined is None else combined + cf
    combined = combined.tocsr()
    combined.data = combined.data * (k1 + 1.0) / (combined.data + k1)
    combined.data *= idf[combined.indices]
    return combined


def haversine(lat1, lon1, lat2, lon2):
    radius = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * radius * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


FIELD_WEIGHTS = (6.0, 1.5, 1.0)
K1 = 0.8
B_PER_FIELD = (0.8, 0.8, 0.8)
print("параметры BM25F зафиксированы")

параметры BM25F зафиксированы


In [12]:
vec, counts, idf = build_index(corpus)
D = bm25f_matrix(counts, FIELD_WEIGHTS, idf, K1, B_PER_FIELD)
del counts
gc.collect()
Dt = D.T.tocsc()
del D
gc.collect()

q_text = (pd.Series(VAL_QTEXT).fillna("").str.lower() + " "
          + pd.Series(VAL_QPARAMS).fillna("").str.lower())
Q = vec.transform(q_text)
Q.data[:] = 1.0
print(f"индекс стенда построен: {len(vec.vocabulary_):,} термов")

индекс стенда построен: 179,433 термов


Поиск ведём блоками запросов, забирая топ-500 кандидатов. 

In [13]:
n_q = len(VAL_TRUTH)
npos = np.array([len(t) for t in VAL_TRUTH], dtype=np.float64)
POOL = 500

glob_cand = np.empty((n_q, POOL), dtype=np.int32)
glob_score = np.empty((n_q, POOL), dtype=np.float32)

ILOC = corpus.item_location_id.values
ILAT = corpus.item_latitude.values
ILON = corpus.item_longitude.values

order_loc = np.argsort(ILOC, kind="stable")
loc_sorted = ILOC[order_loc]
uniq_loc, starts = np.unique(loc_sorted, return_index=True)
ends = np.append(starts[1:], len(loc_sorted))
loc_rows = {int(l): order_loc[s:e] for l, s, e in zip(uniq_loc, starts, ends)}

centroid = corpus.groupby("item_location_id")[["item_latitude", "item_longitude"]].median()
centroid.columns = ["lat", "lon"]
cen_loc = centroid.reindex(uniq_loc)
LOC_LAT, LOC_LON = cen_loc.lat.values, cen_loc.lon.values

K_LOCAL_MAX = 500
loc_cand = np.full((n_q, K_LOCAL_MAX), -1, dtype=np.int32)
loc_score = np.full((n_q, K_LOCAL_MAX), -np.inf, dtype=np.float32)

q_cen = centroid.reindex(VAL_QLOC)
QLAT, QLON = q_cen.lat.values.copy(), q_cen.lon.values.copy()

emp_cen = train.assign(
    lat=pd.to_numeric(train.item_latitude, errors="coerce"),
    lon=pd.to_numeric(train.item_longitude, errors="coerce"),
).groupby("search_location_id")[["lat", "lon"]].median()
missing = ~np.isfinite(QLAT)
rescued = emp_cen.reindex(np.asarray(VAL_QLOC)[missing])
QLAT_RESCUED, QLON_RESCUED = QLAT.copy(), QLON.copy()
QLAT_RESCUED[missing] = rescued.lat.values
QLON_RESCUED[missing] = rescued.lon.values

R_LOCAL = 50.0
for s in range(0, n_q, 128):
    e = min(s + 128, n_q)
    sc = np.asarray((Q[s:e] @ Dt).todense(), dtype=np.float32)
    part = np.argpartition(-sc, kth=POOL - 1, axis=1)[:, :POOL]
    vals = np.take_along_axis(sc, part, 1)
    o = vals.argsort(axis=1)[:, ::-1]
    glob_cand[s:e] = np.take_along_axis(part, o, 1)
    glob_score[s:e] = np.take_along_axis(vals, o, 1)
    for j in range(e - s):
        i = s + j
        dl_ = np.nan_to_num(haversine(QLAT_RESCUED[i], QLON_RESCUED[i], LOC_LAT, LOC_LON),
                            nan=1e6)
        sel = np.where(dl_ <= R_LOCAL)[0]
        if len(sel) == 0:
            continue
        idx = np.concatenate([loc_rows[int(uniq_loc[t])] for t in sel])
        v = sc[j][idx]
        kk = min(K_LOCAL_MAX, len(idx))
        pp = np.argpartition(-v, kth=kk - 1)[:kk] if kk < len(idx) else np.arange(len(idx))
        vv = v[pp]
        oo = vv.argsort()[::-1]
        loc_cand[i, :kk] = idx[pp][oo]
        loc_score[i, :kk] = vv[oo]
    del sc
del Dt, Q
gc.collect()
print("кандидаты обоих каналов отобраны")

кандидаты обоих каналов отобраны


Метрика и вспомогательные функции оценки.

In [14]:
def is_positive_matrix(cand_rows):
    out = np.zeros(cand_rows.shape, dtype=bool)
    for i in range(cand_rows.shape[0]):
        valid = cand_rows[i] >= 0
        out[i, valid] = np.isin(ROW2ITEM[cand_rows[i, valid]], list(VAL_TRUTH[i]))
    return out


def recall_at_k(is_pos, final_score, k=50):
    order = np.argsort(-final_score, axis=1)[:, :k]
    return float((np.take_along_axis(is_pos, order, 1).sum(1) / npos).mean())


ispos_glob = is_positive_matrix(glob_cand)
dist_glob = np.nan_to_num(
    haversine(QLAT[:, None], QLON[:, None], ILAT[glob_cand], ILON[glob_cand]),
    nan=1e6).astype(np.float32)
amp_glob = glob_score.std(axis=1, keepdims=True)

results = {}
results["BM25F без гео"] = recall_at_k(ispos_glob, glob_score)
print(f"BM25F без гео: Recall@50 = {results['BM25F без гео']:.4f}")

BM25F без гео: Recall@50 = 0.4879


## 6. Гео: фильтр или приор

Проверим на стенде два способа использовать локацию. Жёсткий вариант — оставить только
объявления из локации запроса. Мягкий — добавить к скору надбавку, экспоненциально убывающую с
расстоянием до центроида локации запроса.

Гипотеза: жёсткий фильтр проиграет, потому что упирается в потолок 83%, отмеченный в разделе 3.

In [15]:
same_loc_mask = ILOC[glob_cand] == np.asarray(VAL_QLOC)[:, None]
hard = np.where(same_loc_mask, glob_score, -np.inf)
results["гео как жёсткий фильтр"] = recall_at_k(ispos_glob, hard)

geo_bonus = np.exp(-dist_glob / 100.0)
for amp in (2.0, 5.0, 20.0):
    results[f"гео как мягкий приор, {amp:.0f}*std"] = recall_at_k(
        ispos_glob, glob_score + amp * amp_glob * geo_bonus)

for name in ["BM25F без гео", "гео как жёсткий фильтр",
             "гео как мягкий приор, 2*std", "гео как мягкий приор, 5*std",
             "гео как мягкий приор, 20*std"]:
    print(f"  {name:32s}: {results[name]:.4f}")

  BM25F без гео                   : 0.4879
  гео как жёсткий фильтр          : 0.6893
  гео как мягкий приор, 2*std     : 0.6822
  гео как мягкий приор, 5*std     : 0.7456
  гео как мягкий приор, 20*std    : 0.7444


Гипотеза подтвердилась, но с уточнением:. При слабой надбавке
(2·std) мягкий приор жёсткому фильтру проигрывает — 0.682 против 0.689. Выигрыш появляется
только при достаточной амплитуде: на 5·std получается 0.746, то есть примерно на 0.06 выше
фильтра. Иначе говоря, дело было не в самой идее приора, а в том, насколько сильно он задан.

Причина преимущества та же, что и в разделе 3: отсечение безвозвратно теряет клики за пределами
локации, а надбавка их сохраняет, просто опуская ниже по списку. Дальше используем 5·std;
увеличение до 20·std уже ничего не добавляет.

Гео оказывается самым сильным сигналом в задаче — он поднимает метрику примерно в полтора раза
относительно чистой лексики (0.488 против 0.746).

## 7. Запросы без локации в корпусе

У 17% запросов локация не встречается ни у одного объявления корпуса, поэтому центроид для них
не вычисляется и гео-бонус равен нулю. Но эти локации есть в train как локации поиска.
Значит можно построить эмпирический центроид: медиану координат объявлений, которые реально
кликали пользователи, искавшие из этой локации.

In [16]:
print(f"запросов без корпусного центроида: {int(missing.sum())} "
      f"({missing.mean():.1%})")
print(f"из них удалось восстановить по истории train: "
      f"{int(np.isfinite(QLAT_RESCUED[missing]).sum())}")

dist_rescued = np.nan_to_num(
    haversine(QLAT_RESCUED[:, None], QLON_RESCUED[:, None], ILAT[glob_cand], ILON[glob_cand]),
    nan=1e6).astype(np.float32)
bonus_rescued = np.exp(-dist_rescued / 100.0)

base_scores = glob_score + 5.0 * amp_glob * geo_bonus
resc_scores = glob_score + 5.0 * amp_glob * bonus_rescued
results["+ центроиды из истории train"] = recall_at_k(ispos_glob, resc_scores)

order_b = np.argsort(-base_scores, axis=1)[:, :50]
order_r = np.argsort(-resc_scores, axis=1)[:, :50]
per_b = np.take_along_axis(ispos_glob, order_b, 1).sum(1) / npos
per_r = np.take_along_axis(ispos_glob, order_r, 1).sum(1) / npos
print()
print(f"общий Recall@50 : {per_b.mean():.4f} -> {per_r.mean():.4f}")
print(f"на этих запросах: {per_b[missing].mean():.4f} -> {per_r[missing].mean():.4f}")

запросов без корпусного центроида: 427 (17.4%)
из них удалось восстановить по истории train: 427


общий Recall@50 : 0.7456 -> 0.7686


на этих запросах: 0.5946 -> 0.7268


Приём работает: на затронутых запросах метрика растёт заметно, и это транслируется в общий
прирост. Дальше эмпирические центроиды используются всегда.

## 8. Где находится потолок

К этому моменту схема такая: BM25F отбирает топ-500 кандидатов по всему корпусу, затем гео-приор
их переупорядочивает. Вопрос: сколько релевантных объявлений вообще есть в отобранном пуле?

Если позитив не попал в топ-500, никакое ранжирование его уже не вернёт. Доля позитивов,
присутствующих в пуле, — это жёсткий потолок всей конструкции.

In [17]:
pool_ceiling = float((ispos_glob.sum(1) / npos).mean())
current = results["+ центроиды из истории train"]
print(f"потолок пула (доля позитивов внутри топ-500) : {pool_ceiling:.4f}")
print(f"текущий результат                            : {current:.4f}")
print()
print(f"запас на любое улучшение ранжирования        : {pool_ceiling - current:+.4f}")
print(f"запас за пределами пула                      : {1.0 - pool_ceiling:+.4f}")
print(f"гео извлекает {current / pool_ceiling:.1%} того, что в пуле есть")

потолок пула (доля позитивов внутри топ-500) : 0.8071
текущий результат                            : 0.7686

запас на любое улучшение ранжирования        : +0.0385
запас за пределами пула                      : +0.1929
гео извлекает 95.2% того, что в пуле есть


Идеальный реранкер внутри текущего пула добавил бы меньше
четырёх процентных пунктов, тогда как за пределами пула лежит впятеро больше. Гео уже извлекает
около 95% содержимого пула, то есть ранжирование близко к исчерпанию.

Отсюда следует, что дальше нужно менять не порядок кандидатов, а их состав. Посмотрим, что общего
у позитивов, которые в пул не попали.

In [18]:
item2row = {v: i for i, v in enumerate(ROW2ITEM)}
pool_sets = [set(ROW2ITEM[glob_cand[i]]) for i in range(n_q)]
miss_rows, miss_q = [], []
for i in range(n_q):
    for it in VAL_TRUTH[i] - pool_sets[i]:
        miss_rows.append(item2row[it])
        miss_q.append(i)
miss_rows, miss_q = np.array(miss_rows), np.array(miss_q)

d_miss = haversine(QLAT_RESCUED[miss_q], QLON_RESCUED[miss_q], ILAT[miss_rows], ILON[miss_rows])
same = ILOC[miss_rows] == np.asarray(VAL_QLOC)[miss_q]
print(f"позитивов всего {int(npos.sum()):,}, потеряно пулом {len(miss_rows):,} "
      f"({len(miss_rows) / npos.sum():.1%})")
print()
print(f"  в той же локации, что запрос : {same.mean():.1%}")
for r in (10, 25, 50):
    print(f"  в пределах {r:>2} км            : {np.nanmean(d_miss <= r):.1%}")

позитивов всего 2,712, потеряно пулом 521 (19.2%)

  в той же локации, что запрос : 82.3%
  в пределах 10 км            : 80.2%
  в пределах 25 км            : 93.7%
  в пределах 50 км            : 94.8%


Более 80% потерянных позитивов лежат в локации запроса, при том что на локацию запроса
приходится около одного процента корпуса.

Вот в чём дефект конструкции: пул отбирается глобально по лексике, а ранжируется по географии.
Эти две стадии смотрят на разные признаки. Объявление, которое стоит на семисотом месте
глобального лексического списка, но находится в нужном городе, теряется — хотя гео-приор
поставил бы его в первую десятку.

## 9. Второй канал отбора

Исправление прямое: добавить второй канал поиска, который ищет там, куда потом всё равно смотрит
ранжирование. То есть прогнать тот же BM25F, но ограничив кандидатов окрестностью запроса, и
объединить результат с глобальным пулом.

Это изменение кандидатогенерации, а не ранжирования, поэтому оно способно сдвинуть метрику.

Размер локального пула выбираем осторожно. Чем он больше, тем выше потолок, но тем сильнее
локальные объявления с максимальным гео-бонусом вытесняют из топ-50 верные глобальные. Проверим
оба эффекта раздельно.

In [19]:
def build_union(k_local):
    width = POOL + k_local
    u = np.full((n_q, width), -1, dtype=np.int32)
    s = np.full((n_q, width), -np.inf, dtype=np.float32)
    for i in range(n_q):
        r_ = np.concatenate([glob_cand[i], loc_cand[i, :k_local]])
        v_ = np.concatenate([glob_score[i], loc_score[i, :k_local]])
        ok = r_ >= 0
        r_, v_ = r_[ok], v_[ok]
        uq, first = np.unique(r_, return_index=True)
        u[i, :len(uq)] = r_[first]
        s[i, :len(uq)] = v_[first]
    return u, s


def evaluate_union(u, s):
    ip = is_positive_matrix(u)
    uc = np.clip(u, 0, None)
    dd = np.nan_to_num(
        haversine(QLAT_RESCUED[:, None], QLON_RESCUED[:, None], ILAT[uc], ILON[uc]),
        nan=1e6).astype(np.float32)
    mask = np.isfinite(s)
    amp = np.nan_to_num(np.nanstd(np.where(mask, s, np.nan), axis=1, keepdims=True), nan=0.0)
    fin = np.where(mask, s + 5.0 * amp * np.exp(-dd / 100.0), -np.inf)
    return float((ip.sum(1) / npos).mean()), recall_at_k(ip, fin)


print(f"{'конфигурация':34s}{'потолок пула':>14s}{'Recall@50':>12s}")
print(f"{'только глобальный пул':34s}{pool_ceiling:>14.4f}{current:>12.4f}")
for k_local in (100, 200, 500):
    u, s = build_union(k_local)
    ceil_, r50 = evaluate_union(u, s)
    results[f"два канала, K={k_local}"] = r50
    print(f"{'+ локальный канал, K=' + str(k_local):34s}{ceil_:>14.4f}{r50:>12.4f}")
    del u, s
    gc.collect()

конфигурация                        потолок пула   Recall@50
только глобальный пул                     0.8071      0.7686


+ локальный канал, K=100                  0.9366      0.8673


+ локальный канал, K=200                  0.9562      0.8702


+ локальный канал, K=500                  0.9695      0.8640


Второй канал даёт основной прирост всего решения: примерно плюс десять процентных пунктов
на стенде. Потолок пула поднимается с 0.81 до 0.95 и выше.

Обратите внимание на расхождение двух столбцов при K=500: потолок пула продолжает расти, а
Recall@50 уже снижается. Это и есть ожидавшееся вытеснение — лишние локальные кандидаты с
максимальным гео-бонусом выдавливают из топ-50 верные глобальные. Поэтому оптимален средний
размер локального пула, а не максимальный, и именно поэтому обе величины меряются раздельно:
по одному лишь потолку был бы выбран K=500.

Берём K=200.

## 10. Что проверялось и было отвергнуто

Ниже перечислены направления, которые честно измерялись на этом же стенде и в решение не вошли.
Они здесь потому, что именно их отрицательный результат обосновал выбранный путь.

| подход | результат | почему отвергнут |
|---|---|---|
| Жёсткий фильтр по локации | ниже мягкого приора (раздел 6) | упирается в потолок 83% |
| Плотный поиск, rubert-tiny2 без дообучения | Recall@50 = 0.018 | без дообучения эмбеддинги хуже поиска по заголовку |
| Он же после дообучения на парах запрос-объявление | 0.241 отдельно | всё равно втрое хуже лексики с гео |
| Слияние лексики и плотного поиска (RRF, взвешенное) | не выше лексики | пересечение найденного мало: у плотной модели своих находок около 2% |
| Классификатор микрокатегории как бонус к ранжированию | 0.759 -> 0.759 | реранжирование закрытого пула не возвращает отсутствующие объявления |
| Рейтинг, число отзывов, цена как бонус | максимум +0.000 | не разводит почти одинаковые объявления |
| Символьные 3-4-граммы вместо лемм | 0.277 против 0.298 | шумнее лемматизации, слияние не помогло |
| Снижение b в BM25F до 0.3-0.5 по описанию | -0.004 ... -0.006 | найденное значение 0.8 оказалось настоящим оптимумом |

Общий вывод из этой таблицы: пока пул кандидатов содержал лишь 80% достижимого, все попытки
улучшить ранжирование делили между собой очень маленький запас. Это объясняет, почему сразу
несколько технически корректных подходов не дали ничего, и почему в итоге сработало изменение
не ранжирования, а отбора кандидатов.

## 11. Сборка ответа

In [20]:
R_LOCAL_FINAL = 50.0
K_LOCAL_FINAL = 200
GEO_AMP = 5.0
GEO_SCALE_KM = 100.0

bi_work = bi.copy()
for c in ["item_latitude", "item_longitude"]:
    bi_work[c] = pd.to_numeric(bi_work[c], errors="coerce").astype("float64")

vec_f, counts_f, idf_f = build_index(bi_work)
D_f = bm25f_matrix(counts_f, FIELD_WEIGHTS, idf_f, K1, B_PER_FIELD)
del counts_f
gc.collect()
Dt_f = D_f.T.tocsc()
del D_f
gc.collect()

q_text_f = (bq.search_query.fillna("") + " " + bq.search_infm_params_text.fillna("")).str.lower()
Q_f = vec_f.transform(q_text_f)
Q_f.data[:] = 1.0
print(f"индекс бенчмарка построен: {len(vec_f.vocabulary_):,} термов")

индекс бенчмарка построен: 177,631 термов


In [21]:
ILOC_F = bi_work.item_location_id.values
ILAT_F = bi_work.item_latitude.values
ILON_F = bi_work.item_longitude.values

order_f = np.argsort(ILOC_F, kind="stable")
sorted_f = ILOC_F[order_f]
uniq_f, st_f = np.unique(sorted_f, return_index=True)
en_f = np.append(st_f[1:], len(sorted_f))
loc_rows_f = {int(l): order_f[a:b] for l, a, b in zip(uniq_f, st_f, en_f)}

centroid_f = bi_work.groupby("item_location_id")[["item_latitude", "item_longitude"]].median()
centroid_f.columns = ["lat", "lon"]
cen_f = centroid_f.reindex(uniq_f)
LOC_LAT_F, LOC_LON_F = cen_f.lat.values, cen_f.lon.values

q_cen_f = centroid_f.reindex(bq.search_location_id.values)
QLAT_F, QLON_F = q_cen_f.lat.values.copy(), q_cen_f.lon.values.copy()
miss_f = ~np.isfinite(QLAT_F)
resc_f = emp_cen.reindex(bq.search_location_id.values[miss_f])
QLAT_F[miss_f] = resc_f.lat.values
QLON_F[miss_f] = resc_f.lon.values
print(f"запросов без центроида в корпусе: {int(miss_f.sum())}, "
      f"восстановлено по train: {int(np.isfinite(QLAT_F[miss_f]).sum())}")

n_qf = len(bq)
gc_f = np.empty((n_qf, POOL), dtype=np.int32)
gs_f = np.empty((n_qf, POOL), dtype=np.float32)
lc_f = np.full((n_qf, K_LOCAL_FINAL), -1, dtype=np.int32)
ls_f = np.full((n_qf, K_LOCAL_FINAL), -np.inf, dtype=np.float32)

for s in range(0, n_qf, 128):
    e = min(s + 128, n_qf)
    sc = np.asarray((Q_f[s:e] @ Dt_f).todense(), dtype=np.float32)
    part = np.argpartition(-sc, kth=POOL - 1, axis=1)[:, :POOL]
    vals = np.take_along_axis(sc, part, 1)
    o = vals.argsort(axis=1)[:, ::-1]
    gc_f[s:e] = np.take_along_axis(part, o, 1)
    gs_f[s:e] = np.take_along_axis(vals, o, 1)
    for j in range(e - s):
        i = s + j
        dl_ = np.nan_to_num(haversine(QLAT_F[i], QLON_F[i], LOC_LAT_F, LOC_LON_F), nan=1e6)
        sel = np.where(dl_ <= R_LOCAL_FINAL)[0]
        if len(sel) == 0:
            continue
        idx = np.concatenate([loc_rows_f[int(uniq_f[t])] for t in sel])
        v = sc[j][idx]
        kk = min(K_LOCAL_FINAL, len(idx))
        pp = np.argpartition(-v, kth=kk - 1)[:kk] if kk < len(idx) else np.arange(len(idx))
        vv = v[pp]
        oo = vv.argsort()[::-1]
        lc_f[i, :kk] = idx[pp][oo]
        ls_f[i, :kk] = vv[oo]
    del sc
del Dt_f, Q_f
gc.collect()
print("кандидаты отобраны")

запросов без центроида в корпусе: 427, восстановлено по train: 427


кандидаты отобраны


In [22]:
width = POOL + K_LOCAL_FINAL
U_f = np.full((n_qf, width), -1, dtype=np.int32)
S_f = np.full((n_qf, width), -np.inf, dtype=np.float32)
for i in range(n_qf):
    r_ = np.concatenate([gc_f[i], lc_f[i]])
    v_ = np.concatenate([gs_f[i], ls_f[i]])
    ok = r_ >= 0
    r_, v_ = r_[ok], v_[ok]
    uq, first = np.unique(r_, return_index=True)
    U_f[i, :len(uq)] = r_[first]
    S_f[i, :len(uq)] = v_[first]
print(f"средний размер объединённого пула: {int(np.mean((U_f >= 0).sum(1)))}")

Uc_f = np.clip(U_f, 0, None)
dist_f = np.nan_to_num(
    haversine(QLAT_F[:, None], QLON_F[:, None], ILAT_F[Uc_f], ILON_F[Uc_f]),
    nan=1e6).astype(np.float32)
mask_f = np.isfinite(S_f)
amp_f = np.nan_to_num(np.nanstd(np.where(mask_f, S_f, np.nan), axis=1, keepdims=True), nan=0.0)
final_f = np.where(mask_f, S_f + GEO_AMP * amp_f * np.exp(-dist_f / GEO_SCALE_KM), -np.inf)

order_top = np.argsort(-final_f, axis=1)[:, :50]
top50_rows = np.take_along_axis(U_f, order_top, 1)
assert (top50_rows >= 0).all()
top50_items = bi_work.item_id.values[top50_rows]
answer = pd.DataFrame({"query_id": bq.query_id.values,
                       "answer": [" ".join(row) for row in top50_items]})

средний размер объединённого пула: 650


In [23]:
assert len(answer) == len(bq)
assert answer.query_id.is_unique
assert set(answer.query_id) == set(bq.query_id)
assert (answer.query_id.str.len() == 16).all()

valid_items = set(bi.item_id)
for ans in answer.answer:
    ids = ans.split(" ")
    assert len(ids) <= 50
    assert len(ids) == len(set(ids))
    assert all(len(x) == 16 for x in ids)
    assert all(x in valid_items for x in ids)

answer.to_csv("answer.csv", index=False, encoding="utf-8")
print("все проверки формата пройдены")
print(f"answer.csv записан: {len(answer):,} строк, по {top50_items.shape[1]} id в каждой")
answer.head()

все проверки формата пройдены
answer.csv записан: 2,452 строк, по 50 id в каждой


,query_id,answer
0,70DfDUpwjxB4lzFd,d722bcda1a555091 355392014208b7bf 14d535285758af89 255fb...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 422d3ffdd5bbf626 1c6d2079e07f4497 47b2e...
2,LZCZNoVG4AFUkVRJ,d8fce513e4f000a7 3b370cc603f67947 168a9207e80b0be4 dab52...
3,660ac9QVtXkRxZC3,67b2cd847669eb36 af91ec4a29b66636 da18e5119a944f0b 2babc...
4,YgHcM9MVbxKnxD1e,1a62634394544781 53f9c8caeeb8111e 615c73ea4c3bfd15 9c41f...


## Итог

Финальное решение — BM25F по заголовку, параметрам и описанию с лемматизацией, мягкий гео-приор
по расстоянию и двухканальный отбор кандидатов (глобальный поиск плюс поиск в окрестности
запроса). 

Результат на скрытом бенчмарке: Recall@50 = 0.8194.

Главный вывод по ходу работы оказался не про модели, а про диагностику. Несколько технически
корректных улучшений ранжирования — плотный поиск, гибридное слияние, классификатор
микрокатегории, признаки качества объявления — не дали ничего. Причина выяснилась только после
того, как был измерен потолок пула кандидатов: он составлял 0.81, а ранжирование уже вытягивало
0.77 из него. Все эти подходы боролись за остаток менее четырёх процентных пунктов.

Как только выяснилось, что более 80% недостающих объявлений находятся в локации запроса и просто
не попадают в глобальный лексический топ-500, решение свелось к добавлению второго канала
отбора. Это дало основной прирост и не потребовало ни нейросетей, ни дополнительных данных.